# Initial EDA - SaaS Subscription Dataset

This notebook validates the analytical and business quality of the generated MVP dataset before deeper KPI, churn and cohort analysis.

Scope of this notebook:
- load the 5 raw CSV files
- run a first data quality review
- compute key MVP KPIs
- check a few expected business gaps
- confirm whether the dataset is credible enough for the next analysis steps

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

DATA_DIR = Path('../data/raw')
DATASET_END = pd.Timestamp('2025-12-31')


## 2. Load Data

In [ ]:
customers = pd.read_csv(DATA_DIR / 'customers.csv', parse_dates=['signup_date'])
subscriptions = pd.read_csv(
    DATA_DIR / 'subscriptions.csv',
    parse_dates=['trial_start_date', 'trial_end_date', 'subscription_start_date', 'subscription_end_date'],
)
payments = pd.read_csv(DATA_DIR / 'payments.csv', parse_dates=['payment_date'])
product_events = pd.read_csv(DATA_DIR / 'product_events.csv', parse_dates=['event_date'])
monthly_customer_activity = pd.read_csv(
    DATA_DIR / 'monthly_customer_activity.csv',
    parse_dates=['activity_month'],
)

tables = {
    'customers': customers,
    'subscriptions': subscriptions,
    'payments': payments,
    'product_events': product_events,
    'monthly_customer_activity': monthly_customer_activity,
}

for name, frame in tables.items():
    print(f'{name}: {frame.shape[0]:,} rows x {frame.shape[1]} columns')

## 3. Table Overview

In [ ]:
overview = pd.DataFrame(
    [
        {
            'table': name,
            'rows': frame.shape[0],
            'columns': frame.shape[1],
            'memory_mb': round(frame.memory_usage(deep=True).sum() / 1024**2, 2),
        }
        for name, frame in tables.items()
    ]
)
overview

In [ ]:
for name, frame in tables.items():
    print(f'\n{name.upper()}')
    display(frame.head(3))
    display(frame.dtypes.rename('dtype').to_frame())

## 4. Missing Values and Key Uniqueness

In [ ]:
missing_values = pd.concat(
    {
        name: frame.isna().sum().rename('missing_values')
        for name, frame in tables.items()
    },
    axis=1,
).fillna(0).astype(int)
missing_values

In [ ]:
key_checks = pd.DataFrame(
    [
        {'table': 'customers', 'key': 'customer_id', 'is_unique': customers['customer_id'].is_unique},
        {'table': 'subscriptions', 'key': 'subscription_id', 'is_unique': subscriptions['subscription_id'].is_unique},
        {'table': 'payments', 'key': 'payment_id', 'is_unique': payments['payment_id'].is_unique},
        {'table': 'product_events', 'key': 'event_id', 'is_unique': product_events['event_id'].is_unique},
        {
            'table': 'monthly_customer_activity',
            'key': '(customer_id, activity_month)',
            'is_unique': not monthly_customer_activity.duplicated(['customer_id', 'activity_month']).any(),
        },
    ]
)
key_checks

## 5. Simple Chronology Checks

In [ ]:
customer_signup = customers.set_index('customer_id')['signup_date']
subscription_lookup = subscriptions.set_index('customer_id')

chronology_checks = pd.DataFrame(
    [
        {
            'check': 'event_date >= signup_date',
            'passed': bool((product_events['event_date'] >= product_events['customer_id'].map(customer_signup)).all()),
        },
        {
            'check': 'payment_date >= subscription_start_date',
            'passed': bool((payments['payment_date'] >= payments['customer_id'].map(subscription_lookup['subscription_start_date'])).all()),
        },
        {
            'check': 'trial_start_date == signup_date',
            'passed': bool((subscriptions['trial_start_date'] == subscriptions['customer_id'].map(customer_signup)).all()),
        },
        {
            'check': 'trial_end_date >= trial_start_date',
            'passed': bool((subscriptions['trial_end_date'] >= subscriptions['trial_start_date']).all()),
        },
        {
            'check': 'converted customers have subscription_start_date',
            'passed': bool(subscriptions.loc[subscriptions['converted_to_paid'], 'subscription_start_date'].notna().all()),
        },
    ]
)
chronology_checks

## 6. Global Distributions

In [ ]:
display(customers['acquisition_channel'].value_counts(normalize=True).mul(100).round(1).rename('share_pct').to_frame())
display(subscriptions['subscription_status'].value_counts(normalize=True).mul(100).round(1).rename('share_pct').to_frame())
display(subscriptions.loc[subscriptions['converted_to_paid'], 'plan_type'].value_counts(normalize=True).mul(100).round(1).rename('share_pct').to_frame())
display(product_events['event_type'].value_counts(normalize=True).mul(100).round(1).rename('share_pct').to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

customers['acquisition_channel'].value_counts().sort_values().plot(kind='barh', ax=axes[0], color='#3b82f6')
axes[0].set_title('Customers by Acquisition Channel')
axes[0].set_xlabel('Customers')

subscriptions['subscription_status'].value_counts().plot(kind='bar', ax=axes[1], color='#10b981')
axes[1].set_title('Subscription Status Distribution')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Count')

product_events['event_type'].value_counts().sort_values().plot(kind='barh', ax=axes[2], color='#f59e0b')
axes[2].set_title('Product Events Distribution')
axes[2].set_xlabel('Events')

plt.tight_layout()
plt.show()

## 7. Activation Definition and MVP KPIs

Activation rule used here:
- within 7 days after signup
- at least one of `bank_account_connected` or `transaction_imported`
- and at least one `budget_created`

In [ ]:
activation_events = product_events.merge(customers[['customer_id', 'signup_date']], on='customer_id', how='left')
activation_events = activation_events[
    activation_events['event_date'] <= activation_events['signup_date'] + pd.Timedelta(days=6)
]

activation_status = (
    activation_events.groupby('customer_id')['event_type']
    .agg(list)
    .apply(
        lambda events: (
            any(event in {'bank_account_connected', 'transaction_imported'} for event in events)
            and 'budget_created' in events
        )
    )
    .rename('is_activated')
)

customers_enriched = customers.merge(activation_status, on='customer_id', how='left')
customers_enriched['is_activated'] = customers_enriched['is_activated'].eq(True)

subscriptions_enriched = subscriptions.merge(
    customers_enriched[['customer_id', 'acquisition_channel', 'is_activated']],
    on='customer_id',
    how='left',
)

active_paid_end = subscriptions[
    subscriptions['converted_to_paid']
    & (subscriptions['subscription_start_date'] <= DATASET_END)
    & (
        subscriptions['subscription_end_date'].isna()
        | (subscriptions['subscription_end_date'] > DATASET_END)
    )
]

mrr_end_period = active_paid_end['monthly_price'].sum()
paid_payments = payments[payments['payment_status'] == 'paid']
arpu_simple = paid_payments['amount'].sum() / subscriptions.loc[subscriptions['converted_to_paid'], 'customer_id'].nunique()

monthly_subs = subscriptions[(subscriptions['converted_to_paid']) & (subscriptions['plan_type'] == 'monthly')].copy()
effective_end = monthly_subs['subscription_end_date'].fillna(DATASET_END)
active_months = (
    effective_end.dt.to_period('M') - monthly_subs['subscription_start_date'].dt.to_period('M')
).apply(lambda period_delta: period_delta.n + 1)
monthly_churn_approx = monthly_subs['subscription_status'].eq('cancelled').sum() / active_months.clip(lower=1).sum()

kpis = pd.DataFrame(
    [
        {'kpi': 'Customers', 'value': len(customers)},
        {'kpi': 'Activation rate', 'value': customers_enriched['is_activated'].mean()},
        {'kpi': 'Trial to paid conversion rate', 'value': subscriptions['converted_to_paid'].mean()},
        {
            'kpi': 'Annual plan share among paid customers',
            'value': subscriptions.loc[subscriptions['converted_to_paid'], 'plan_type'].eq('annual').mean(),
        },
        {'kpi': 'Approximate end-of-period MRR', 'value': mrr_end_period},
        {'kpi': 'Simple ARPU', 'value': arpu_simple},
        {'kpi': 'Approximate monthly churn (monthly plans)', 'value': monthly_churn_approx},
    ]
)

kpis

## 8. Expected Business Gaps

Interpretation note for usage-based churn:
- `monthly_customer_activity` contains observed months with events only
- it does not include explicit zero-activity months
- as a result, the latest observed usage segment is useful as a directional validation signal, but not as a final churn metric

In [ ]:
conversion_by_channel = (
    subscriptions_enriched.groupby('acquisition_channel')['converted_to_paid']
    .mean()
    .sort_values(ascending=False)
    .rename('conversion_rate')
    .to_frame()
)

conversion_by_activation = (
    subscriptions_enriched.groupby('is_activated')['converted_to_paid']
    .mean()
    .rename('conversion_rate')
    .to_frame()
)

churn_by_plan = (
    subscriptions.loc[subscriptions['converted_to_paid']]
    .groupby('plan_type')['subscription_status']
    .apply(lambda status: status.eq('cancelled').mean())
    .rename('churn_rate')
    .to_frame()
)

# monthly_customer_activity contains only observed months with events, not explicit zero-activity months.
latest_usage = (
    monthly_customer_activity.sort_values(['customer_id', 'activity_month'])
    .groupby('customer_id')
    .tail(1)[['customer_id', 'usage_segment']]
)

usage_vs_churn = (
    subscriptions[['customer_id', 'subscription_status', 'converted_to_paid']]
    .merge(latest_usage, on='customer_id', how='left')
)
usage_vs_churn = usage_vs_churn[usage_vs_churn['converted_to_paid']].copy()
usage_vs_churn['usage_segment'] = usage_vs_churn['usage_segment'].fillna('no_observed_activity')

churn_by_usage = (
    usage_vs_churn.groupby('usage_segment')['subscription_status']
    .apply(lambda status: status.eq('cancelled').mean())
    .sort_values()
    .rename('churn_rate')
    .to_frame()
)

display(conversion_by_channel)
display(conversion_by_activation)
display(churn_by_plan)
display(churn_by_usage)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

conversion_by_channel['conversion_rate'].sort_values().plot(kind='barh', ax=axes[0, 0], color='#2563eb')
axes[0, 0].set_title('Conversion Rate by Acquisition Channel')
axes[0, 0].set_xlabel('Conversion rate')

conversion_by_activation['conversion_rate'].rename(index={False: 'Not activated', True: 'Activated'}).plot(
    kind='bar', ax=axes[0, 1], color=['#f97316', '#16a34a']
)
axes[0, 1].set_title('Conversion Rate: Activated vs Not Activated')
axes[0, 1].set_xlabel('Activation status')
axes[0, 1].set_ylabel('Conversion rate')

churn_by_plan['churn_rate'].plot(kind='bar', ax=axes[1, 0], color=['#0f766e', '#dc2626'])
axes[1, 0].set_title('Churn Rate by Plan')
axes[1, 0].set_xlabel('Plan type')
axes[1, 0].set_ylabel('Churn rate')

churn_by_usage['churn_rate'].plot(kind='bar', ax=axes[1, 1], color='#7c3aed')
axes[1, 1].set_title('Churn Rate by Latest Observed Usage Segment')
axes[1, 1].set_xlabel('Usage segment')
axes[1, 1].set_ylabel('Churn rate')

plt.tight_layout()
plt.show()

## 9. Final Validation Notes

In [ ]:
summary = [
    f"Customers: {len(customers):,}",
    f"Activation rate: {customers_enriched['is_activated'].mean():.1%}",
    f"Conversion rate: {subscriptions['converted_to_paid'].mean():.1%}",
    f"Annual share among paid customers: {subscriptions.loc[subscriptions['converted_to_paid'], 'plan_type'].eq('annual').mean():.1%}",
    f"Approximate monthly churn on monthly plans: {monthly_churn_approx:.1%}",
    f"End-of-period MRR: {mrr_end_period:,.2f}",
]

print('What looks coherent')
for item in summary:
    print(f'- {item}')

print('\nWhat deserves attention')
print('- monthly_customer_activity contains observed months with events only, not explicit zero-activity months')
print('- churn and usage checks are good for validation, but final KPI analysis should recompute metrics from business definitions')

print('\nEDA V1 conclusion')
print('- the dataset is analytically usable for KPI analysis, churn analysis and simple cohort analysis')
print('- the next step can focus on business KPIs rather than data generation issues')